# Analyze example

This notebook just loads the response files from LLMs and inspects what they look like and how the models performed

In [1]:
import json
import re
import pandas as pd
from foodscraper.config import *
from datetime import date

In [2]:
responses_path = DATA_DIR / "runs" / "2026-09-07" / "responses"

In [3]:
response_files = sorted(responses_path.glob("*.csv"))
df = pd.concat((pd.read_csv(f) for f in response_files), ignore_index=True)


In [4]:
# the `answer` column is a JSON blob (sometimes wrapped in prose or a ```json fence) -> unfold it
def extract_json(text):
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    i, j = text.find("{"), text.rfind("}")
    if i != -1 and j > i:
        try:
            return json.loads(text[i:j + 1])
        except Exception:
            pass
    return {}

parsed = df["answer"].apply(extract_json)
results = pd.concat(
    [df[["model", "restaurant_id"]].reset_index(drop=True), pd.json_normalize(parsed)],
    axis=1,
)
results["model"] = results["model"].str.split("/").str[-1]   # anthropic/claude-opus-5 -> claude-opus-5
results[["model", "name", "hasFoundMenu", "menuLink", "type", "source", "notes"]]

,model,name,hasFoundMenu,menuLink,type,source,notes
0,claude-opus-5,Berlin Döner,True,https://wolt.com/en/dnk/copenhagen/restaurant/...,HTML,WOLT,Checked the official site berlindoner.dk (most...
1,claude-opus-5,Ata Pizza,True,https://ata-pizza.dk/,HTML,OFFICIAL_WEBSITE,"Checked the official site (ata-pizza.dk), whic..."
2,claude-opus-5,Sticks'n'Sushi - Restaurant Vesterbro,True,https://www.sticksnsushi.com/dk/en/dine-in-menu/,HTML,OFFICIAL_WEBSITE,The restaurant's own page for Istedgade links ...
3,claude-opus-5,Skipper's Bodega,False,,,,"Checked the listed Facebook page, Google/aggre..."
4,claude-opus-5,Auténtica Twuería Mexicana,False,,,,"No websiteUri was provided (NaN), and searches..."
...,...,...,...,...,...,...,...
150,gpt-5.6-terra,Bar la Una,True,https://menu02.restaurantguru.com/m2/menu-Bar-...,JPEG,OTHER_THIRD_PARTY,Restaurant Guru displays a JPEG menu identifie...
151,gpt-5.6-terra,Bootleggers Vesterbro,True,https://bootleggers.dk/wp-content/uploads/2024...,PDF,OFFICIAL_WEBSITE,Found an official Bootleggers PDF drink menu h...
152,gpt-5.6-terra,Madhuset,True,https://www.madhusetvesterbro.dk/,HTML,OFFICIAL_WEBSITE,The official Madhuset Vesterbro website contai...
153,gpt-5.6-terra,Star Midnight Kebab-Grill,True,https://menukort.menu/restaurants/koebenhavn/s...,HTML,OTHER_THIRD_PARTY,A third-party Menukort.menu page for the match...


In [5]:
models = sorted(results["model"].unique())

# per-model performance at finding menus
summary = (
    results.groupby("model")
    .agg(
        n=("hasFoundMenu", "size"),
        found=("hasFoundMenu", "sum"),
        types=("type", lambda s: ", ".join(sorted(s.dropna().unique()))),
        sources=("source", lambda s: ", ".join(sorted(s.dropna().unique()))),
    )
    .assign(found_rate=lambda d: d["found"] / d["n"])
    .sort_values("found_rate", ascending=False)
)
summary

,n,found,types,sources,found_rate
model,,,,,
gpt-5.6-luna,31,25,", HTML, JPEG, PDF",", OFFICIAL_WEBSITE, OTHER_THIRD_PARTY, UBEREAT...",0.806452
gpt-5.5,31,24,", HTML, JPEG, PDF",", OFFICIAL_WEBSITE, OTHER_THIRD_PARTY, UBEREAT...",0.774194
gpt-5.6-terra,31,23,", HTML, JPEG, PDF",", OFFICIAL_WEBSITE, OTHER_THIRD_PARTY, WOLT",0.741935
claude-opus-5,31,21,", HTML",", OFFICIAL_WEBSITE, OTHER_THIRD_PARTY, WOLT",0.677419
claude-sonnet-4-6,31,19,", HTML",", OFFICIAL_WEBSITE, OTHER_THIRD_PARTY, WOLT",0.612903


In [6]:
# side-by-side: what menu link each model picked for every restaurant
comparison = results.pivot_table(
    index="name", columns="model", values="menuLink", aggfunc="first"
)
comparison

model,claude-opus-5,claude-sonnet-4-6,gpt-5.5,gpt-5.6-luna,gpt-5.6-terra
name,,,,,
1001 Nat Pizza,https://www.dagensmenu.dk/takeaway/star1001-na...,https://menuweb.menu/restaurants/koebenhavn/10...,https://weur-cdn.menukort.menu/storage/media/c...,https://weur-cdn.menukort.menu/storage/media/c...,https://weur-cdn.menukort.menu/storage/media/c...
Ankara Durum House,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://www.ubereats.com/dk/store/ankara-durum...,https://www.ubereats.com/dk-en/store/ankara-du...,https://wolt.com/en/dnk/copenhagen/restaurant/...
Asia Cooking,,,,,
Ata Pizza,https://ata-pizza.dk/,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://atapizza.dk/takeaway,https://atapizza.dk/pickup,https://atapizza.dk/pickup
Auténtica Twuería Mexicana,,,,,
Bar la Una,,,https://www.baruna.dk/s/1JulyFoodUna.pdf,https://menu02.restaurantguru.com/m2/menu-Bar-...,https://menu02.restaurantguru.com/m2/menu-Bar-...
Berlin Döner,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...
Bevi Bevi,,,https://www.bevibevi.dk/s/18MarchDrinkBevi-rh8...,https://weur-cdn.menukort.menu/storage/media/c...,https://www.bevibevi.dk/s/18MarchDrinkBevi-rh8...
Bootleggers Vesterbro,https://untappd.com/v/bootleggers-vesterbro/13...,,https://bootleggers.dk/wp-content/uploads/2024...,https://untappd.com/v/bootleggers-vesterbro/13...,https://bootleggers.dk/wp-content/uploads/2024...


In [7]:
# all menu links found by the best-performing model (gpt-5.6-luna)
luna = results[results["model"] == "gpt-5.6-luna"].copy()
luna = luna[luna["hasFoundMenu"].astype(bool) & luna["menuLink"].fillna("").str.strip().ne("")]

luna_menus = luna[["name", "menuLink", "type", "source"]].reset_index(drop=True)
luna_links = luna_menus["menuLink"].tolist()

print(f"{len(luna_links)} menu links from gpt-5.6-luna")
for link in luna_links:
    print(link)

luna_menus

25 menu links from gpt-5.6-luna
https://wolt.com/en/dnk/copenhagen/restaurant/berlin-dner-kopenhagen
https://atapizza.dk/pickup
https://a.storyblok.com/f/286316/x/a7b0fc6a7f/sns-menu-dk-eng-2026-05-210x210-web.pdf
https://menu02.restaurantguru.com/m7/menu-Cafe-Erik.jpg
https://www.laneta.dk/menu
https://istedgrill.dk/menu/
https://www.ubereats.com/dk-en/store/ankara-durum-house/k27UBtaTUBqWKtdMPLdg3Q
https://gaodumpling.com/istedgade
https://laforetta.bestilonline.dk/
https://wolt.com/en/dnk/copenhagen/restaurant/mezze-cph
https://wolt.com/en/dnk/copenhagen/restaurant/ankara-durum-house
https://pizzeriamamemi.dk/en/menu/?lang=en
https://pashakebab.dk/menu/
https://grillazcph.dk/?scroll-to-menu=true
https://cafepatina.dk/menu/
https://grimal.dk/madogdrikke/menu
https://static1.squarespace.com/static/67d980ea39323d366cf40dae/t/69a94e5860550043223952bd/1772703320827/MENUKORT_DANSK.pdf
https://www.paparamen.dk/menu?menu=papa-ramen
https://weur-cdn.menukort.menu/storage/media/companies_menu

,name,menuLink,type,source
0,Berlin Döner,https://wolt.com/en/dnk/copenhagen/restaurant/...,HTML,WOLT
1,Ata Pizza,https://atapizza.dk/pickup,HTML,OFFICIAL_WEBSITE
2,Sticks'n'Sushi - Restaurant Vesterbro,https://a.storyblok.com/f/286316/x/a7b0fc6a7f/...,PDF,OFFICIAL_WEBSITE
3,Cafe Erik,https://menu02.restaurantguru.com/m7/menu-Cafe...,JPEG,OTHER_THIRD_PARTY
4,La Neta Vesterbro,https://www.laneta.dk/menu,HTML,OFFICIAL_WEBSITE
5,Isted Grill,https://istedgrill.dk/menu/,HTML,OFFICIAL_WEBSITE
6,Ankara Durum House,https://www.ubereats.com/dk-en/store/ankara-du...,HTML,UBEREATS
7,GAO Dumpling Bar,https://gaodumpling.com/istedgade,HTML,OFFICIAL_WEBSITE
8,La Foretta,https://laforetta.bestilonline.dk/,HTML,OFFICIAL_WEBSITE
9,Mezze,https://wolt.com/en/dnk/copenhagen/restaurant/...,HTML,WOLT


## Where do the models agree / disagree on `menuLink`?

Same idea as `manual_comparer.ipynb`, generalized to all 5 models here. Each restaurant is
classified by comparing the (normalized) `menuLink` every model returned for it:

- `all_not_found` — no model found a menu
- `all_agree` — every model found a menu, and it's the same link
- `some_found_agree` — only some models found a menu, but the ones that did agree on the link
- `some_found_disagree` — only some models found a menu, and they disagree on the link
- `all_disagree` — every model found *a* menu, but at different links

In [8]:
def norm(url):
    if not isinstance(url, str) or not url.strip():
        return None
    return url.strip().rstrip("/").lower()


def classify(row):
    vals = [norm(row[m]) for m in models]
    found = [v for v in vals if v is not None]
    if not found:
        return "all_not_found"
    unique = set(found)
    if len(found) == len(vals):
        return "all_agree" if len(unique) == 1 else "all_disagree"
    return "some_found_agree" if len(unique) == 1 else "some_found_disagree"


menu_comparison = results.pivot_table(
    index="name", columns="model", values="menuLink", aggfunc="first"
).reindex(columns=models)
menu_comparison["status"] = menu_comparison.apply(classify, axis=1)

menu_comparison["status"].value_counts()

status
all_disagree           17
all_not_found           5
some_found_disagree     4
some_found_agree        3
all_agree               2
Name: count, dtype: int64

In [9]:
# restaurants where the models disagree (or split) on the menu link
disagreements = menu_comparison[
    menu_comparison["status"].isin(["some_found_disagree", "all_disagree"])
].sort_values("status")

disagreements

model,claude-opus-5,claude-sonnet-4-6,gpt-5.5,gpt-5.6-luna,gpt-5.6-terra,status
name,,,,,,
1001 Nat Pizza,https://www.dagensmenu.dk/takeaway/star1001-na...,https://menuweb.menu/restaurants/koebenhavn/10...,https://weur-cdn.menukort.menu/storage/media/c...,https://weur-cdn.menukort.menu/storage/media/c...,https://weur-cdn.menukort.menu/storage/media/c...,all_disagree
Ankara Durum House,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://www.ubereats.com/dk/store/ankara-durum...,https://www.ubereats.com/dk-en/store/ankara-du...,https://wolt.com/en/dnk/copenhagen/restaurant/...,all_disagree
Ata Pizza,https://ata-pizza.dk/,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://atapizza.dk/takeaway,https://atapizza.dk/pickup,https://atapizza.dk/pickup,all_disagree
Berlin Döner,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/da/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,all_disagree
Börger,https://www.burger-vesterbro.dk/en/menu,https://www.burger-vesterbro.dk/en/menu,https://www.burger-vesterbro.dk/menu,https://www.burger-vesterbro.dk/menu,https://www.burger-vesterbro.dk/menu,all_disagree
Café Patina,https://cafepatina.dk/en/menu-2/,https://cafepatina.dk/en/menu-2/,https://cafepatina.dk/menu/,https://cafepatina.dk/menu/,https://cafepatina.dk/menu/,all_disagree
GAO Dumpling Bar,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://wolt.com/en/dnk/copenhagen/restaurant/...,https://gaodumpling.com/istedgade,https://gaodumpling.com/istedgade,https://gaodumpling.com/istedgade,all_disagree
Grillaz,https://grillazcph.dk/,https://grillazcph.dk/,https://grillazcph.dk/?scroll-to-menu=true,https://grillazcph.dk/?scroll-to-menu=true,https://grillazcph.dk/?scroll-to-menu=true,all_disagree
Grimal,https://grimal.dk/en/menu/menu,https://grimal.dk/en/menu/menu,https://grimal.dk/madogdrikke/menu,https://grimal.dk/madogdrikke/menu,https://grimal.dk/en/menu/menu,all_disagree


## Pairwise agreement between models

For each pair of models, restrict to restaurants where *both* found a menu link and compute
how often they picked the same (normalized) link. This isolates disagreement in link choice
from disagreement in whether a menu exists at all.

In [10]:
import itertools

normalized = menu_comparison[models].apply(lambda col: col.map(norm))

pairwise_agreement = pd.DataFrame(index=models, columns=models, dtype=float)
for m1, m2 in itertools.combinations_with_replacement(models, 2):
    both_found = normalized[m1].notna() & normalized[m2].notna()
    n_both = both_found.sum()
    n_agree = (normalized.loc[both_found, m1] == normalized.loc[both_found, m2]).sum()
    rate = n_agree / n_both if n_both else float("nan")
    pairwise_agreement.loc[m1, m2] = rate
    pairwise_agreement.loc[m2, m1] = rate

pairwise_agreement.round(2)

,claude-opus-5,claude-sonnet-4-6,gpt-5.5,gpt-5.6-luna,gpt-5.6-terra
claude-opus-5,1.00,0.74,0.19,0.24,0.33
claude-sonnet-4-6,0.74,1.00,0.11,0.21,0.26
gpt-5.5,0.19,0.11,1.00,0.57,0.65
gpt-5.6-luna,0.24,0.21,0.57,1.00,0.74
gpt-5.6-terra,0.33,0.26,0.65,0.74,1.00
